# Phase 3 — Self-Training (CBST) + Fine-tuning (Colab)

Picks up from the Phase 2 pretrained 9-class checkpoint and runs:
1. **Pseudo-labeling** of BaDLAD-unlabeled with the pretrained model
2. **CBST** class-balanced thresholds (rare classes not swamped)
3. **Self-training rounds** on {real + pseudo}
4. **Fine-tune on IndicDLP** (re-headed to IndicDLP's ontology) — *added next*

Logic lives in `src/finetuning/self_training.py`; these cells only orchestrate.
**Runtime:** T4 is fine for the smoke test (Cell 4). Switch to **A100** for the full run (Cell 5), off-peak.

## Cell 0 — Bootstrap (mount Drive, install deps)

In [ ]:
# Runtime -> Change runtime type -> A100 (full run) or T4 (smoke test) -> Save FIRST
import os, sys, subprocess
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive', force_remount=True)
PROJECT_ROOT = Path('/content/drive/MyDrive/doclayout-yolo-indic')
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(PROJECT_ROOT))

subprocess.run(['pip','install','-q','ultralytics','huggingface_hub','pyyaml',
                'pycocotools','kagglehub','tqdm'], check=True)
subprocess.run(['pip','install','-q',
                'git+https://github.com/opendatalab/DocLayout-YOLO.git'], check=True)

import torch
print('PyTorch :', torch.__version__)
print('CUDA    :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU     :', torch.cuda.get_device_name(0))
print('PROJECT_ROOT:', PROJECT_ROOT)

## Cell 1 — Sync code from GitHub (plain files, no zip)
Clones the repo, auto-detects `src/` (works whether it's at the repo root or inside a
wrapper dir like `doclayout-yolo-indic/`), and copies it to `PROJECT_ROOT/src` on Drive so
`config.py`'s `__file__`-anchored paths resolve to Drive (persistent outputs/checkpoints).
**Prereq:** commit `src/` as plain files (drop the zip) and add
`src/finetuning/self_training.py` + `__init__.py`.

In [ ]:
import subprocess, shutil
from pathlib import Path

GITHUB_URL    = 'https://github.com/vigneshpalanivelr/mtech-project-aiml.git'
GITHUB_BRANCH = 'main'
REPO_LOCAL    = Path('/content/_repo')

shutil.rmtree(REPO_LOCAL, ignore_errors=True)
subprocess.run(['git','clone','--depth','1','-b',GITHUB_BRANCH,
                GITHUB_URL, str(REPO_LOCAL)], check=True)

# Find src/ wherever it lives: repo root, or one level down (wrapper dir).
candidates = [REPO_LOCAL/'src'] + sorted(REPO_LOCAL.glob('*/src'))
SRC = next((c for c in candidates if (c/'config.py').exists()), None)
assert SRC, f'Could not find src/config.py under {REPO_LOCAL}'
BASE = SRC.parent
print('Found project at:', BASE)

# Copy code (not docs) to PROJECT_ROOT so REPO_ROOT = parents[1] -> Drive.
for item in ['src','tests','requirements.txt','README.md']:
    s = BASE/item; d = PROJECT_ROOT/item
    if s.is_dir():   shutil.rmtree(d, ignore_errors=True); shutil.copytree(s, d)
    elif s.exists(): shutil.copy(s, d)

assert (PROJECT_ROOT/'src'/'finetuning'/'self_training.py').exists(), \
    'Commit src/finetuning/self_training.py + __init__.py to the repo first!'
print('Code synced (plain files) ->', PROJECT_ROOT/'src')

## Cell 2 — Download BaDLAD (Kaggle: reasat/badlad-train)
4 classes: text-box, paragraph, image, table. Re-pulled each session (don't persist to Drive).
**Prereq:** add `KAGGLE_USERNAME` + `KAGGLE_KEY` to Colab Secrets (left key panel).

In [ ]:
import os
from google.colab import userdata
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY']      = userdata.get('KAGGLE_KEY')

import kagglehub
BADLAD_PATH = kagglehub.dataset_download('reasat/badlad-train')
print('BaDLAD cached at:', BADLAD_PATH)
for root, dirs, files in os.walk(BADLAD_PATH):
    print(root, '->', len(files), 'files')
    if root != BADLAD_PATH: break
# Set this to the folder that actually holds the .jpg/.png pages:
BADLAD_IMAGES = BADLAD_PATH  # adjust after inspecting the printout above

## Cell 3 — Stage IndicDLP from your private HF repo
Pull only the splits you need. Upload it once from local first (see chat instructions).

In [ ]:
from pathlib import Path
from huggingface_hub import snapshot_download, login
from google.colab import userdata

login(token=userdata.get('HF_TOKEN'))
INDICDLP_DIR = Path('/content/IndicDLP')
if INDICDLP_DIR.exists() and any(INDICDLP_DIR.rglob('*.json')):
    print('IndicDLP already staged at', INDICDLP_DIR)
else:
    snapshot_download(repo_id='VigneshPR/indicdlp', repo_type='dataset',
                      local_dir=str(INDICDLP_DIR),
                      allow_patterns=['train/*','val/*','*.json','*.yaml'])
    print('IndicDLP staged ->', INDICDLP_DIR)

## Cell 4 — Smoke test: CBST pseudo-labeling on ~50 images (T4 OK)
Verifies the *novel* part — inference + class-balanced thresholds + label writing — fast, no full train.

In [ ]:
%cd {PROJECT_ROOT}
from src.finetuning.self_training import (load_detector, collect_predictions,
    compute_cbst_thresholds, filter_with_thresholds, write_pseudo_labels)

PRETRAINED = PROJECT_ROOT/'output'/'checkpoints'/'doclayout_yolo_indic_pretrained.pt'

model, names = load_detector(PRETRAINED)
recs = collect_predictions(model, BADLAD_IMAGES, device=0, limit=50)
thr  = compute_cbst_thresholds(recs, num_classes=len(names), portion=0.20)
kept, stats = filter_with_thresholds(recs, thr)

print('CBST thresholds :', {names[k]: round(v,3) for k,v in thr.items()})
print('kept/total/class:', {names[int(k)]: v for k,v in stats.items()})
write_pseudo_labels(kept, '/content/pseudo_smoke', names)
print(f'OK -- wrote {len(kept)} pseudo-label files. CBST pipeline works.')

## Cell 5 — Full self-training (A100, off-peak)
Two CBST rounds, resumable. `LABELED=None` runs pseudo-only now; once `data_prep.py`
builds the BaDLAD 9-class yaml, point `LABELED` at it for the supervised half.

In [ ]:
%cd {PROJECT_ROOT}
from src.finetuning.self_training import run_self_training

PRETRAINED = PROJECT_ROOT/'output'/'checkpoints'/'doclayout_yolo_indic_pretrained.pt'
LABELED    = None  # or PROJECT_ROOT/'data'/'raw'/'BaDLAD'/'badlad_9class.yaml'

final_best = run_self_training(
    pretrained   = PRETRAINED,
    unlabeled_dir= BADLAD_IMAGES,
    labeled_yaml = LABELED,
    work_dir     = PROJECT_ROOT/'output'/'self_training',
    num_rounds   = 2,
    device       = 0)
print('Final self-training checkpoint:', final_best)

## Next
- `src/finetuning/data_prep.py` — BaDLAD COCO->YOLO 9-class remap + IndicDLP yaml
- `src/finetuning/train_finetuning.py` — re-head to IndicDLP's 42 classes + fine-tune (final model)
- `src/finetuning/ablation.py` — the 3-5 ablation runs

(`src/utils/ontology.py` already exists: pretrain 9-class head, then **replace the head**
with IndicDLP's 42 classes at fine-tuning. Run its `inspect_indicdlp_categories()` first.)